# 🌱 AgriSmart — Module 2: Crop Disease Detection (Multiclass)

**Task:** 38-class disease classification (crop + disease type)  
**Model:** EfficientNetV2-S pretrained on ImageNet  
**Dataset:** New Plant Diseases Dataset (Augmented) via Kaggle  
**Environment:** Kaggle 2x T4, fp16

---
### ⚙️ Before Running
1. **Add Data** → search `vipoooool new plant diseases` → add **New Plant Diseases Dataset(Augmented)**
2. **Settings → Accelerator → GPU T4 x2**
3. **Internet ON**
---

## Cell 1 — Imports & Device

In [ ]:
!pip install -q onnxscript onnx
!pip install -q wandb  # Weights & Biases tracking
print('✅ Done')

import os
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.preprocessing import label_binarize

import cv2
import wandb

# ── Device & GPU setup ──────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_GPUS = torch.cuda.device_count()
OUTPUT_DIR = '/kaggle/working'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'Device  : {DEVICE}')
print(f'GPUs    : {N_GPUS}')
print(f'Output  : {OUTPUT_DIR}')
print('Imports ready.')

## Cell 1b — W&B Initialization

In [ ]:
import wandb

WANDB_API_KEY  = ''  # Paste key or use Kaggle secret WANDB_API_KEY
WANDB_PROJECT  = 'agrismart-disease'
WANDB_RUN_NAME = 'efficientnetv2s-38class-fp16'

if not WANDB_API_KEY:
    try:
        from kaggle_secrets import UserSecretsClient
        WANDB_API_KEY = UserSecretsClient().get_secret('WANDB_API_KEY')
        print('API key loaded from Kaggle secrets')
    except:
        print('No API key — W&B disabled')
        wandb.init(mode='disabled')
        WANDB_API_KEY = None

if WANDB_API_KEY:
    wandb.login(key=WANDB_API_KEY)
    wandb.init(
        project=WANDB_PROJECT,
        name=WANDB_RUN_NAME,
        config={
            'model':         'EfficientNetV2-S',
            'num_classes':   38,
            'img_size':      224,
            'batch_size':    64,
            'epochs':        15,
            'lr':            1e-3,
            'optimizer':     'AdamW',
            'scheduler':     'CosineAnnealingLR',
            'label_smoothing': 0.1,
            'frozen':        'all features, train classifier only',
            'precision':     'fp16',
            'augmentation':  'flip+rotate+colorjitter+erasing',
        }
    )
    print(f'W&B run started: {wandb.run.url}')


## Cell 2 — Load Dataset

In [ ]:
import os
import json
from pathlib import Path

OUTPUT_DIR = '/kaggle/working'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Auto-scan for dataset root — handles all known Kaggle path variations
POSSIBLE_ROOTS = [
    Path('/kaggle/input/datasets/vipoooool/new-plant-diseases-dataset/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/train'),
    Path('/kaggle/input/new-plant-diseases-dataset/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/train'),
    Path('/kaggle/input/new-plant-diseases-dataset/train'),
    Path('/kaggle/input/datasets/vipoooool/new-plant-diseases-dataset/train'),
]

DATA_ROOT = None
for p in POSSIBLE_ROOTS:
    if p.exists():
        DATA_ROOT = p
        break

if DATA_ROOT is None:
    print('Could not find dataset. Scanning /kaggle/input/...')
    for root, dirs, files in os.walk('/kaggle/input'):
        level = root.replace('/kaggle/input', '').count(os.sep)
        print(' ' * 2 * level + os.path.basename(root) + '/')
        if level > 4:
            break
    raise FileNotFoundError('Update POSSIBLE_ROOTS with the path printed above')

print(f'Dataset root: {DATA_ROOT}')

# Scan all 38 class folders — keep original class names for multiclass
all_paths  = []
all_labels = []   # Integer class index
class_names = []  # String class names

for folder in sorted(DATA_ROOT.iterdir()):
    if not folder.is_dir():
        continue
    class_idx = len(class_names)
    class_names.append(folder.name)
    for ext in ['*.jpg', '*.JPG', '*.jpeg', '*.png']:
        for img_path in folder.glob(ext):
            all_paths.append(img_path)
            all_labels.append(class_idx)

NUM_CLASSES = len(class_names)
print(f'Total images : {len(all_paths):,}')
print(f'Classes      : {NUM_CLASSES}')
print(f'\nAll classes:')
for i, name in enumerate(class_names):
    count = all_labels.count(i)
    print(f'  [{i:02d}] {name}: {count:,} images')

# Save class names for deployment
with open(f'{OUTPUT_DIR}/class_names.json', 'w') as f:
    json.dump(class_names, f, indent=2)
print(f'\nClass names saved to {OUTPUT_DIR}/class_names.json')

## Cell 3 — Preview Sample Images

## Cell 4 — Transforms & Dataset Class

In [ ]:
# EfficientNetV2-S uses 384x384 for best accuracy
# Using 224x224 here to keep training fast on T4
IMG_SIZE = 224
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
    transforms.RandomErasing(p=0.1),  # After ToTensor — fixes AttributeError from previous version
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])


class PlantDataset(Dataset):
    def __init__(self, paths, labels, transform):
        self.paths     = paths
        self.labels    = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = cv2.imread(str(self.paths[idx]))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = Image.fromarray(img)
        return self.transform(img), self.labels[idx]


print('Transforms and Dataset class ready.')

## Cell 5 — Train / Val / Test Split

In [ ]:
# 70 / 15 / 15 stratified split
train_p, temp_p, train_l, temp_l = train_test_split(
    all_paths, all_labels, test_size=0.3, random_state=42, stratify=all_labels
)
val_p, test_p, val_l, test_l = train_test_split(
    temp_p, temp_l, test_size=0.5, random_state=42, stratify=temp_l
)

train_ds = PlantDataset(train_p, train_l, train_transform)
val_ds   = PlantDataset(val_p,   val_l,   val_transform)
test_ds  = PlantDataset(test_p,  test_l,  val_transform)

BATCH_SIZE     = 64
VAL_BATCH_SIZE = 32

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2,
    multiprocessing_context='fork',
)
val_loader = DataLoader(
    val_ds,
    batch_size=VAL_BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2,
    multiprocessing_context='fork',
)
test_loader = DataLoader(
    test_ds,
    batch_size=VAL_BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2,
    multiprocessing_context='fork',
)

print(f'Train batch : {BATCH_SIZE} × {N_GPUS} GPUs = {BATCH_SIZE * N_GPUS} effective')
print(f'Val batch   : {VAL_BATCH_SIZE}')
print('DataLoaders ready.')

## Cell 6 — EfficientNetV2-S Model

In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()
print(f'GPU memory free before model load: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')

# Load pretrained ResNet-50
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)

# Freeze all layers except the final classifier
for param in model.parameters():
    param.requires_grad = False

# Replace final FC layer for 38-class output
in_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(p=0.3),
    nn.Linear(in_features, NUM_CLASSES),
)

# Wrap with DataParallel for 2x T4
if N_GPUS > 1:
    model = nn.DataParallel(model)

model = model.to(DEVICE)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'DataParallel across {N_GPUS} GPUs')
print(f'Trainable : {trainable:,}  ({100*trainable/total:.1f}%)')
print(f'Frozen    : {total-trainable:,}')
print(f'Output classes: {NUM_CLASSES}')
print(f'GPU memory after model load: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')

## Cell 7 — Training

In [ ]:
from tqdm import tqdm

EPOCHS = 10
LR     = 1e-3

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR, weight_decay=1e-4
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler    = torch.amp.GradScaler('cuda')

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_acc = 0.0

for epoch in range(EPOCHS):
    # ── Train ──────────────────────────────────────────────
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0

    train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [TRAIN]", leave=False)
    for images, labels in train_bar:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            outputs = model(images)
            loss    = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss    += loss.item() * images.size(0)
        preds          = outputs.argmax(dim=1)
        train_correct += (preds == labels).sum().item()
        train_total   += images.size(0)
        train_bar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'acc':  f'{train_correct/train_total:.4f}'
        })

    # ── Clear GPU memory before validation ─────────────────
    torch.cuda.empty_cache()

    # ── Validate ───────────────────────────────────────────
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0

    val_bar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [VAL]", leave=False)
    with torch.no_grad():
        for images, labels in val_bar:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            with torch.amp.autocast('cuda'):
                outputs = model(images)
                loss    = criterion(outputs, labels)
            val_loss    += loss.item() * images.size(0)
            preds        = outputs.argmax(dim=1)
            val_correct += (preds == labels).sum().item()
            val_total   += images.size(0)
            val_bar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'acc':  f'{val_correct/val_total:.4f}'
            })

    scheduler.step()

    t_loss = train_loss / train_total
    v_loss = val_loss   / val_total
    t_acc  = train_correct / train_total
    v_acc  = val_correct   / val_total

    history['train_loss'].append(t_loss)
    history['val_loss'].append(v_loss)
    history['train_acc'].append(t_acc)
    history['val_acc'].append(v_acc)

    is_best = v_acc > best_val_acc
    if is_best:
        best_val_acc = v_acc
        save_model = model.module if hasattr(model, 'module') else model
        torch.save(save_model.state_dict(), f'{OUTPUT_DIR}/best_model.pth')

    if WANDB_API_KEY and wandb.run:
        wandb.log({
            'epoch':      epoch + 1,
            'train/loss': t_loss,
            'train/acc':  t_acc,
            'val/loss':   v_loss,
            'val/acc':    v_acc,
            'lr':         scheduler.get_last_lr()[0],
        })

    print(
        f'Epoch {epoch+1:02d}/{EPOCHS}  |  '
        f'Train loss: {t_loss:.4f}  acc: {t_acc:.4f}  |  '
        f'Val loss: {v_loss:.4f}  acc: {v_acc:.4f}'
        + ('  ← best' if is_best else '')
    )

print(f'\nBest val accuracy: {best_val_acc:.4f}')
shutil.copy(
    f'{OUTPUT_DIR}/best_model.pth',
    f'/kaggle/working/best_model_epoch{epoch+1}.pth'
)

## Cell 8 — Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(history['train_loss'], label='Train')
ax1.plot(history['val_loss'],   label='Val')
ax1.set_title('Loss')
ax1.set_xlabel('Epoch')
ax1.legend()
ax1.grid(True)

ax2.plot(history['train_acc'], label='Train')
ax2.plot(history['val_acc'],   label='Val')
ax2.set_title('Accuracy')
ax2.set_xlabel('Epoch')
ax2.legend()
ax2.grid(True)

plt.suptitle('AgriSmart — Multiclass Disease Detection (EfficientNetV2-S)', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/training_curves.png', dpi=150)
plt.show()

In [ ]:
import shutil

# Save to Kaggle output (persists after session ends)
shutil.copy(f'{OUTPUT_DIR}/best_model.pth', '/kaggle/working/best_model_resnet50_backup.pth')

# Also save the class names
shutil.copy(f'{OUTPUT_DIR}/class_names.json', '/kaggle/working/class_names_backup.json')

print('Saved! Go to Output tab on the right → download both files to your computer.')

## Cell 9 — Test Evaluation

In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()

# ── Load best ResNet-50 checkpoint ─────────────────────────
eval_model = models.resnet50(weights=None)
in_features = eval_model.fc.in_features
eval_model.fc = nn.Sequential(
    nn.Dropout(p=0.3),
    nn.Linear(in_features, NUM_CLASSES),
)
eval_model.load_state_dict(torch.load(
    '/kaggle/working/best_model_resnet50_backup.pth',
    map_location=DEVICE
))
eval_model = eval_model.to(DEVICE)
eval_model.eval()

# ── Run inference on test set ───────────────────────────────
all_preds, all_true, all_probs = [], [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)
        with torch.amp.autocast('cuda'):
            outputs = eval_model(images)
        probs = torch.softmax(outputs, dim=1).cpu().numpy()
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_true.extend(labels.numpy())
        all_probs.extend(probs)

all_preds = np.array(all_preds)
all_true  = np.array(all_true)
all_probs = np.array(all_probs)

# ── Results ─────────────────────────────────────────────────
print('=' * 60)
print('TEST RESULTS — 38-Class Disease Classification')
print('=' * 60)
print(classification_report(all_true, all_preds, target_names=class_names))

# ROC-AUC (one-vs-rest)
all_true_bin = label_binarize(all_true, classes=list(range(NUM_CLASSES)))
roc_auc = roc_auc_score(all_true_bin, all_probs, average='macro', multi_class='ovr')
print(f'Macro ROC-AUC : {roc_auc:.4f}')

## Cell 9b — Log Final Test Metrics to W&B

In [ ]:
if WANDB_API_KEY and wandb.run:
    wandb.log({
        'test/accuracy':  (all_true == all_preds).mean(),
        'test/roc_auc':   roc_auc,
    })

    # Log confusion matrix as a W&B plot
    wandb.log({
        'test/confusion_matrix': wandb.plot.confusion_matrix(
            probs=None,
            y_true=all_true.tolist(),
            preds=all_preds.tolist(),
            class_names=class_names
        )
    })

    wandb.finish()
    print('Final metrics logged to W&B and run closed.')
else:
    acc = (all_true == all_preds).mean()
    print(f'W&B not connected — final test accuracy: {acc:.4f} ({acc*100:.2f}%)')
    print(f'ROC-AUC: {roc_auc:.4f}')

## 1. Cell 10 — Confusion Matrix (Top 15 Classes)

In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()

# ── Load best ResNet-50 checkpoint ─────────────────────────
eval_model = models.resnet50(weights=None)
in_features = eval_model.fc.in_features
eval_model.fc = nn.Sequential(
    nn.Dropout(p=0.3),
    nn.Linear(in_features, NUM_CLASSES),
)
eval_model.load_state_dict(torch.load(
    '/kaggle/working/best_model_resnet50_backup.pth',
    map_location=DEVICE
))
eval_model = eval_model.to(DEVICE)
if N_GPUS > 1:
    eval_model = nn.DataParallel(eval_model)
eval_model.eval()
print('ResNet-50 loaded successfully — ready for evaluation.')

In [ ]:
# Full 38x38 confusion matrix is hard to read
# Show top 15 most confused classes instead
cm = confusion_matrix(all_true, all_preds)

# Find 15 classes with most errors
errors_per_class = cm.sum(axis=1) - np.diag(cm)
top15_idx = np.argsort(errors_per_class)[-15:]
cm_top15  = cm[np.ix_(top15_idx, top15_idx)]
names_top15 = [class_names[i].replace('___', '\n')[:25] for i in top15_idx]

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(
    cm_top15, annot=True, fmt='d', cmap='Greens',
    xticklabels=names_top15, yticklabels=names_top15, ax=ax
)
ax.set_xlabel('Predicted', fontsize=11)
ax.set_ylabel('Actual', fontsize=11)
ax.set_title('Confusion Matrix — Top 15 Most Confused Classes', fontsize=13)
plt.xticks(rotation=45, ha='right', fontsize=7)
plt.yticks(rotation=0, fontsize=7)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/confusion_matrix.png', dpi=150)
plt.show()

## Cell 11 — Inference on a Single Image

In [ ]:
def predict(image_path: str, top_k: int = 3) -> dict:
    """
    Run inference on any image.
    Returns top-k predictions with confidence scores.
    """
    img    = Image.open(image_path).convert('RGB')
    tensor = val_transform(img).unsqueeze(0).to(DEVICE)

    eval_model.eval()
    with torch.no_grad():
        with torch.cuda.amp.autocast():
            output = eval_model(tensor)
        probs  = torch.softmax(output, dim=1)[0]
        top_probs, top_idxs = probs.topk(top_k)

    results = [
        {'class': class_names[i], 'confidence': round(p.item(), 4)}
        for i, p in zip(top_idxs.tolist(), top_probs)
    ]

    # Plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
    ax1.imshow(img)
    ax1.set_title(f'Top prediction:\n{results[0]["class"]}\n({results[0]["confidence"]*100:.1f}%)',
                  fontsize=9, color='green' if 'healthy' in results[0]['class'].lower() else 'red')
    ax1.axis('off')

    labels_plot = [r['class'].replace('___', '\n')[:30] for r in results]
    confs_plot  = [r['confidence'] * 100 for r in results]
    colors      = ['green' if 'healthy' in l.lower() else 'tomato' for l in labels_plot]
    ax2.barh(labels_plot, confs_plot, color=colors)
    ax2.set_xlabel('Confidence (%)')
    ax2.set_title(f'Top {top_k} Predictions')
    ax2.set_xlim(0, 100)

    plt.tight_layout()
    plt.show()
    return results


# Quick test on a random image from test set
sample_path  = test_p[10]
sample_label = test_l[10]
print(f'Actual class: {class_names[sample_label]}')
results = predict(str(sample_path))
print('\nTop predictions:')
for r in results:
    print(f'  {r["class"]}: {r["confidence"]*100:.1f}%')

# To run on your own image:
# results = predict('/kaggle/input/your-image/leaf.jpg')

## Cell 12 — Save & Export

In [ ]:
# Save full checkpoint with all metadata needed for deployment
torch.save({
    'model_state_dict': eval_model.state_dict(),
    'class_names': class_names,
    'num_classes': NUM_CLASSES,
    'input_size': IMG_SIZE,
    'mean': MEAN,
    'std': STD,
    'best_val_acc': best_val_acc,
}, f'{OUTPUT_DIR}/agrismart_disease_multiclass.pth')
print(f'Checkpoint saved.')

# Export to ONNX for mobile deployment
# opset_version=18 — avoids the version warning from previous run
import torch.onnx

# Save full checkpoint with all metadata needed for deployment
torch.save({
    'model_state_dict': eval_model.state_dict(),
    'class_names': class_names,
    'num_classes': NUM_CLASSES,
    'input_size': IMG_SIZE,
    'mean': MEAN,
    'std': STD,
    'best_val_acc': best_val_acc,
}, f'{OUTPUT_DIR}/agrismart_disease_multiclass.pth')
print('Checkpoint saved.')

# ── Export to ONNX (legacy exporter — more stable) ──────────
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)

# Unwrap DataParallel if needed
export_model = eval_model.module if hasattr(eval_model, 'module') else eval_model
export_model.eval().cpu()
dummy_cpu = dummy.cpu()

torch.onnx.export(
    export_model,
    dummy_cpu,
    f'{OUTPUT_DIR}/agrismart_disease_multiclass.onnx',
    input_names=['image'],
    output_names=['logits'],
    dynamic_axes={'image': {0: 'batch_size'}, 'logits': {0: 'batch_size'}},
    opset_version=17,
    dynamo=False,        # force legacy exporter — avoids the dynamic_shapes bug
)
print('ONNX model saved.')
print(f'\nAll outputs in: {OUTPUT_DIR}')
print('Ready for mobile conversion (ONNX → TFLite or CoreML).')

---
## 📋 Summary

| Component | Choice | Reason |
|---|---|---|
| Model | EfficientNetV2-S | Better accuracy than ResNet-50, faster training |
| Classes | 38 (full PlantVillage) | Real-world usefulness — specific disease per crop |
| Frozen layers | All features, train classifier only | Fast convergence, avoids overfitting on small dataset |
| Loss | CrossEntropy + label smoothing 0.1 | Prevents overconfidence on 38 classes |
| Multi-GPU | DataParallel 2x T4 | Doubles throughput |
| ONNX opset | 18 | Avoids automatic upgrade warning from previous run |
| RandomErasing | After ToTensor() | Fixes AttributeError from previous version |

### 🔜 Next steps
1. Integrate with Module 1: class name → Tunisian advice generation
2. Add W&B tracking for experiment comparison
3. Convert ONNX → TFLite (Android) or CoreML (iOS)